# Bearer Tokens and OAuth2

This notebook covers:

1. Why auth lives at the framework edge, not inside business logic
2. Use the `HTTPBearer` security scheme for the simplest protected route
3. Issue opaque tokens from a `/token` endpoint using `OAuth2PasswordBearer`
4. Propagate the authenticated user through `Depends(get_current_user)`
5. Distinguish 401 (not authenticated) from 403 (authenticated but forbidden)
6. Inspect how the auth flow lands in OpenAPI / `/docs`

**Scope**: FastAPI + `passlib` + `TestClient`. Tokens here are opaque random strings stored in a process-local dict — enough to learn the wiring. JWTs come next in notebook 5.2.

## 1. Auth at the API Layer

Two questions every protected route has to answer:

- **Who are you?** — *authentication*. The answer is an identity, attached to the request.
- **Are you allowed to do this?** — *authorization*. The answer is a yes / no, given an identity and an action.

Conflate them and your error messages become useless ("unauthorized" doesn't say whether the fix is to log in or to ask for a role). Separate them and you get a clean failure model:

- No credentials, or bad credentials → **401 Unauthorized**. The client should authenticate (or re-authenticate) and retry.
- Valid credentials, but the identity isn't allowed → **403 Forbidden**. Retrying with the same credentials won't help.

The HTTP spec is unfortunate here: "unauthorized" reads like "not allowed," but it means "not authenticated." Memorize the mapping and you'll never name it wrong again.

The pattern this notebook builds:

1. A **security scheme** declares *how* the credential travels (header? cookie? scheme name?).
2. A **dependency** extracts that credential from the request and verifies it.
3. The dependency returns the authenticated `User` — or raises 401.
4. Route handlers take the user as a parameter; they never touch headers.

That layering means the auth code lives in one place, and every protected route gets the identity for free.

## 2. HTTP Bearer Tokens

The simplest scheme: client sends `Authorization: Bearer <token>` and the server validates `<token>`. "Bearer" means whoever holds the token can use it — no key exchange, no signing on the client side. The token is the credential.

FastAPI's `fastapi.security.HTTPBearer` parses the header for you. It returns an `HTTPAuthorizationCredentials` object with `.scheme` (`"Bearer"`) and `.credentials` (the token). If the header is missing or malformed, the scheme raises 401 automatically — you don't write that branch yourself.

In [ ]:
from fastapi import Depends, FastAPI, HTTPException, status
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from fastapi.testclient import TestClient

app = FastAPI()
bearer = HTTPBearer()  # the security scheme: "look for Authorization: Bearer ..."

# A toy token registry. Real apps look this up in a database, a cache, or by decoding a JWT (next notebook).
VALID_TOKENS = {"demo-token-abc123": "alice"}

@app.get("/portfolios/me")
def read_my_portfolio(creds: HTTPAuthorizationCredentials = Depends(bearer)):
    username = VALID_TOKENS.get(creds.credentials)
    if username is None:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Invalid token")
    return {"user": username, "holdings": [{"ticker": "AAPL", "shares": 10}]}

client = TestClient(app)

# No header at all -> 401 from the scheme.
print("no header :", client.get("/portfolios/me").status_code)

# Wrong token -> our explicit 401.
print("bad token :", client.get("/portfolios/me", headers={"Authorization": "Bearer wrong"}).status_code)

# Right token -> 200 with body.
ok = client.get("/portfolios/me", headers={"Authorization": "Bearer demo-token-abc123"})
print("good token:", ok.status_code, ok.json())

Three things worth noticing in the output:

- The route function never touches `request.headers`. The `Depends(bearer)` argument *is* the parsed credential.
- A missing or malformed header returns **401** automatically — the scheme raises it before our handler runs. (Older FastAPI versions used 403 here; modern versions consistently use 401 for "no credentials" and reserve 403 for "credentials present but not allowed.")
- Pass `HTTPBearer(auto_error=False)` if you want the dependency to return `None` instead of raising, so your handler can build a custom 401 body. We rarely need that — the default is fine.

## 3. `OAuth2PasswordBearer` Dependency

`HTTPBearer` is the protocol-level scheme. `OAuth2PasswordBearer` is one rung up: it's the OAuth2 *password flow* — "client posts username + password to `/token`, gets back a bearer token, sends it on subsequent requests." It still uses `Authorization: Bearer ...` on the wire, so on the read side it's almost the same thing. The reason to prefer it:

- **OpenAPI integration.** `/docs` renders an "Authorize" button that knows where the token endpoint is (`tokenUrl="/token"`). Click it once, paste credentials, and every "Try it out" call is authenticated. With raw `HTTPBearer`, learners have to paste a header by hand every time.
- **Standard contract.** Any client library that speaks OAuth2 password flow works out of the box.
- **Scopes.** OAuth2 schemes carry a scope vocabulary (`read:portfolios`, `write:portfolios`) we'll wire up in the exercises.

Mechanically: the dependency parses the same header and returns the token *as a string* (not the wrapping `HTTPAuthorizationCredentials`).

In [ ]:
from fastapi.security import OAuth2PasswordBearer

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/token")
# tokenUrl is metadata for OpenAPI. It does NOT auto-create the route — we still build /token in the next section.

demo_app = FastAPI()

@demo_app.get("/whoami")
def whoami(token: str = Depends(oauth2_scheme)):
    # token is the raw string after "Bearer " — verification still ours to do.
    return {"received_token": token}

demo_client = TestClient(demo_app)
r = demo_client.get("/whoami", headers={"Authorization": "Bearer demo-token-abc123"})
print(r.status_code, r.json())

# And the default 401-vs-no-header behavior of OAuth2PasswordBearer:
print("no header:", demo_client.get("/whoami").status_code)  # 401, not 403 — nicer default than HTTPBearer.

One small difference from `HTTPBearer`: `OAuth2PasswordBearer` returns the **bare token string**, not a wrapper object. One less attribute access. Both schemes raise **401** on a missing header (older docs say 403 for `HTTPBearer`; modern FastAPI unified them on 401).

For the rest of the notebook we'll use `OAuth2PasswordBearer` and treat the returned string as opaque — the *contents* of the token are an implementation detail of the issuer (us, in the next section).

## 4. Issuing a Token Endpoint

The OAuth2 password flow expects a `POST /token` that:

- Accepts `username` and `password` as **form fields** (not JSON — that's what the spec says, and what `/docs` will send).
- Verifies the credentials against a user store.
- Returns `{"access_token": "...", "token_type": "bearer"}`.

FastAPI ships `OAuth2PasswordRequestForm` as a dependency that parses the form fields for you. For credential storage we'll use **`passlib`** with PBKDF2-SHA256: never store plaintext passwords, and use a slow KDF so brute force is expensive. (Bcrypt and Argon2 are the other production picks; we use PBKDF2 here because it ships with passlib with no native dependency.)

Token contents in this notebook are a random hex string — "opaque" tokens. Verification works by looking the token up in a server-side map. That's fine for one-server demos and gives a great pedagogical contrast for notebook 5.2, which switches to *self-describing* tokens (JWTs) that need no lookup.

In [ ]:
import secrets
from passlib.context import CryptContext
from pydantic import BaseModel
from fastapi.security import OAuth2PasswordRequestForm

pwd_context = CryptContext(schemes=["pbkdf2_sha256"], deprecated="auto")

class UserInDB(BaseModel):
    username: str
    hashed_password: str
    disabled: bool = False
    roles: list[str] = []  # used in section 5 for 401 vs 403

# A tiny user "database". hash() is slow on purpose — only call it at registration / setup.
USERS: dict[str, UserInDB] = {
    "alice": UserInDB(username="alice", hashed_password=pwd_context.hash("alice-pw"), roles=["trader"]),
    "bob":   UserInDB(username="bob",   hashed_password=pwd_context.hash("bob-pw"),   roles=["viewer"]),
}

# Opaque-token store: token -> username. Lives in process memory only.
TOKENS: dict[str, str] = {}

def authenticate(username: str, password: str) -> UserInDB | None:
    user = USERS.get(username)
    if user is None:
        return None
    if not pwd_context.verify(password, user.hashed_password):
        return None
    return user

auth_app = FastAPI()
auth_scheme = OAuth2PasswordBearer(tokenUrl="/token")

class TokenResponse(BaseModel):
    access_token: str
    token_type: str = "bearer"

@auth_app.post("/token", response_model=TokenResponse)
def login(form: OAuth2PasswordRequestForm = Depends()):
    user = authenticate(form.username, form.password)
    if user is None:
        # Per RFC 6749: the OAuth token endpoint signals bad credentials with 400 + error code,
        # but in practice FastAPI examples use 401 here for consistency with protected routes.
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Incorrect username or password",
            headers={"WWW-Authenticate": "Bearer"},  # tells the client which scheme to use
        )
    token = secrets.token_hex(16)
    TOKENS[token] = user.username
    return TokenResponse(access_token=token)

auth_client = TestClient(auth_app)

# Bad password -> 401.
bad = auth_client.post("/token", data={"username": "alice", "password": "wrong"})
print("bad creds:", bad.status_code, bad.json())

# Good password -> token in the body.
ok = auth_client.post("/token", data={"username": "alice", "password": "alice-pw"})
print("good creds:", ok.status_code, ok.json())

Five details worth highlighting:

- **`data=` not `json=`** on the test client. The OAuth2 password flow uses form encoding (`application/x-www-form-urlencoded`). Send JSON and the form parser sees no fields.
- **`OAuth2PasswordRequestForm`** does the form parsing as a dependency. You get `.username`, `.password`, and — if you wire it up — `.scopes`.
- **`pwd_context.verify`** is constant-time and slow (PBKDF2 is intentionally expensive). Don't compare hashes with `==`.
- **`WWW-Authenticate: Bearer`** in the 401 response. The spec asks for it; many client libraries key off it to decide which auth flow to attempt.
- **No data leak in the error message.** "Incorrect username or password" — not "user not found" vs "wrong password". The username-existence check is itself information.

## 5. Validating Tokens on Every Request (and 401 vs 403)

Now the read side. We build a `get_current_user` dependency that:

1. Pulls the token from the `Authorization` header (via `oauth2_scheme`).
2. Looks it up in `TOKENS`.
3. Loads the corresponding `UserInDB`.
4. Raises **401** if any of those steps fails (no identity).

Then we chain a second dependency, `require_role("trader")`, that runs *after* `get_current_user`. It already has an authenticated user; it just decides whether that user is allowed. Failure mode here is **403**, not 401, because re-authenticating won't change the answer.

This two-layer pattern is the standard FastAPI idiom: one dep produces identity, another dep enforces policy. Each route declares the *minimum* dep it needs.

In [ ]:
def get_current_user(token: str = Depends(auth_scheme)) -> UserInDB:
    username = TOKENS.get(token)
    if username is None:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid or expired token",
            headers={"WWW-Authenticate": "Bearer"},
        )
    user = USERS.get(username)
    if user is None or user.disabled:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="User unavailable")
    return user

def require_role(role: str):
    """Dependency *factory*: returns a Depends-compatible callable that checks for a role."""
    def dep(user: UserInDB = Depends(get_current_user)) -> UserInDB:
        if role not in user.roles:
            raise HTTPException(
                status_code=status.HTTP_403_FORBIDDEN,
                detail=f"Requires role '{role}'",
            )
        return user
    return dep

@auth_app.get("/portfolios/me")
def my_portfolio(user: UserInDB = Depends(get_current_user)):
    # Open to any authenticated user (alice trader, bob viewer).
    return {"user": user.username, "roles": user.roles}

@auth_app.post("/portfolios/me/trades")
def place_trade(user: UserInDB = Depends(require_role("trader"))):
    return {"placed_by": user.username, "status": "accepted"}

# Re-issue tokens for both users so we can demo all three branches.
alice_token = auth_client.post("/token", data={"username": "alice", "password": "alice-pw"}).json()["access_token"]
bob_token   = auth_client.post("/token", data={"username": "bob",   "password": "bob-pw"}).json()["access_token"]

# 1) No header at all -> 401 from oauth2_scheme.
print("no auth /me:           ", auth_client.get("/portfolios/me").status_code)

# 2) Invalid token -> 401 from get_current_user.
print("bad token /me:         ", auth_client.get("/portfolios/me", headers={"Authorization": "Bearer nope"}).status_code)

# 3) Valid token, route is open to any user -> 200.
r = auth_client.get("/portfolios/me", headers={"Authorization": f"Bearer {alice_token}"})
print("alice /me:             ", r.status_code, r.json())

# 4) Valid token, route requires 'trader', alice has it -> 200.
r = auth_client.post("/portfolios/me/trades", headers={"Authorization": f"Bearer {alice_token}"})
print("alice /trades (trader):", r.status_code, r.json())

# 5) Valid token, route requires 'trader', bob is only a viewer -> 403, not 401.
r = auth_client.post("/portfolios/me/trades", headers={"Authorization": f"Bearer {bob_token}"})
print("bob /trades   (viewer):", r.status_code, r.json())

Read the five output lines top to bottom and you have the full failure ladder:

| Situation                              | Status | Where it's raised        |
|----------------------------------------|--------|--------------------------|
| No `Authorization` header              | 401    | `oauth2_scheme`          |
| Unknown / expired token                | 401    | `get_current_user`       |
| Authenticated, route is open           | 200    | (success)                |
| Authenticated, route needs role, has it| 200    | (success)                |
| Authenticated, role missing            | **403** | `require_role(...)`     |

The principle: **the absence of credentials is always 401; the absence of permissions is always 403.** A 403 tells the client "don't bother re-logging in; you simply can't do this."

## 6. OpenAPI Auth Documentation

Declaring the auth scheme as a FastAPI security dependency does two extra things — both visible in `/openapi.json`:

- A `securitySchemes` entry describes the auth flow to clients (and to `/docs`).
- Each protected route gets a `security` entry referencing that scheme. The "lock" icon next to the operation in `/docs` is rendered from this.

Concretely: because we declared `OAuth2PasswordBearer(tokenUrl="/token")`, the generated schema includes an OAuth2 password flow pointed at our `/token` endpoint. When a learner opens `/docs`, the *Authorize* button knows the flow.

In [ ]:
schema = auth_app.openapi()

print("securitySchemes keys :", list(schema["components"]["securitySchemes"].keys()))
print("OAuth2PasswordBearer :", schema["components"]["securitySchemes"]["OAuth2PasswordBearer"])
print()
print("/portfolios/me security entry  :", schema["paths"]["/portfolios/me"]["get"].get("security"))
print("/portfolios/me/trades security :", schema["paths"]["/portfolios/me/trades"]["post"].get("security"))
print("/token (issuer) security       :", schema["paths"]["/token"]["post"].get("security"))
# /token has no security entry — the issuer is intentionally public, since you have no token yet.

The `securitySchemes` block names the scheme (`OAuth2PasswordBearer`) and points at the flow (`tokenUrl: /token`). Every route that depends on `oauth2_scheme` — directly or transitively, via `get_current_user` and `require_role` — inherits a `security: [{OAuth2PasswordBearer: []}]` entry. The empty list is where **scopes** go; we'll wire those up in the exercises.

`/token` itself has *no* security entry. That's correct: the token-issuing endpoint can't require a token. In a real app you'd protect it with rate limiting and a TLS-only deployment instead (notebook 5.3, notebook 8.1).

## Key Takeaways

- **Authentication answers "who"; authorization answers "can."** Map them to **401** and **403** respectively. Never collapse them.
- **Bearer tokens travel in `Authorization: Bearer <token>`.** Use `HTTPBearer` for the raw protocol, `OAuth2PasswordBearer` for the OAuth2 password flow (better `/docs` ergonomics, scope-ready).
- **The `/token` endpoint** takes form-encoded credentials via `OAuth2PasswordRequestForm`, verifies with `passlib` (constant-time, slow on purpose), and returns `{access_token, token_type: "bearer"}`.
- **`get_current_user`** is the canonical FastAPI auth dep: extract token → verify → return user, or raise 401. Compose role checks on top as separate deps that raise 403.
- **Opaque tokens** need a server-side lookup table (`TOKENS`). Fine for one process; useless across replicas. JWTs (notebook 5.2) trade lookup for signature verification.
- **OpenAPI gets it for free.** `securitySchemes` + per-route `security` mean `/docs` shows the lock icon and the Authorize button automatically.
- **Capstone tie-in**: `auth.py` will export `oauth2_scheme`, `get_current_user`, and `require_role` exactly as written here, except the user lookup goes through the repository pattern from 4.2 and the token verification is replaced by JWT decode from 5.2.

## Exercises

**1. Add a protected route and force a 403.** Add `@auth_app.delete("/portfolios/me")` that depends on `require_role("admin")`. Neither `alice` nor `bob` has the admin role. Confirm:

- No header → 401
- Bogus token → 401
- Alice's valid token (trader, not admin) → **403**
- Add `"admin"` to alice's `roles` list and re-issue her token → 200

Write the four assertions out as a single TestClient script.

**2. Scopes, properly.** Change `OAuth2PasswordBearer(tokenUrl="/token", scopes={"read:portfolios": "read", "write:portfolios": "write"})` and update `/token` to honor `form.scopes`. Store the granted scopes alongside the username in `TOKENS` (e.g., as a tuple). Replace `require_role` with `require_scope`, raising 403 if the requested scope isn't on the token. Verify in the `/docs` Authorize dialog that the two scope checkboxes appear.

**3. Logout / revoke.** Add `POST /logout` that deletes the current request's token from `TOKENS`. After calling it, re-using that token must return 401 with `detail="Invalid or expired token"`. Write the round-trip as a single test: log in, hit `/portfolios/me` (200), log out (204), hit `/portfolios/me` again (401). Note in a markdown cell *why* this revocation strategy doesn't survive a server restart and doesn't generalize to multiple replicas — that's the motivation for stateless tokens in notebook 5.2.